# PDF Chunking — pypdf

Reads the extracted text cached by `PDF Parsing (pypdf)` in `rag_chatbot_dev.bronze.doc_parsed_pypdf`, splits it into overlapping chunks, and upserts the chunks into the `rag_chatbot_dev.silver.doc_chunks_pypdf` Delta table for downstream RAG use and side-by-side experimentation with the built-in functions pipeline (`doc_chunks_builtin`).

In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
TEXT_TABLE = f"{CATALOG}.bronze.doc_parsed_pypdf"       # raw text cache written by PDF Parsing (pypdf)
CHUNK_TABLE = f"{CATALOG}.silver.doc_chunks_pypdf"     # target Delta table (pypdf pipeline, silver layer)
CHUNK_SIZE = 1000      # characters per chunk
CHUNK_OVERLAP = 200    # character overlap between consecutive chunks

from pyspark.sql import functions as F

# Text extraction is decoupled: read the cached text instead of re-parsing PDFs
texts_df = spark.table(TEXT_TABLE)
print(f"Loaded {texts_df.count()} document text(s) from {TEXT_TABLE}")
display(texts_df.select("doc_name", F.length("text").alias("text_chars")))

In [0]:
import re

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf


@pandas_udf("array<string>")
def chunk_text(texts: pd.Series) -> pd.Series:
    """Split one document's text into overlapping chunks of fixed size."""
    chunks_out = []
    for text in texts:
        # Collapse whitespace so chunks are clean, contiguous spans of text
        cleaned = re.sub(r"\s+", " ", text or "").strip()

        chunks = []
        start = 0
        while start < len(cleaned):
            chunks.append(cleaned[start : start + CHUNK_SIZE])
            if start + CHUNK_SIZE >= len(cleaned):
                break
            start += CHUNK_SIZE - CHUNK_OVERLAP
        chunks_out.append(chunks)
    return pd.Series(chunks_out)


# Chunk the cached texts, then attach stable ids and metadata
chunks_df = (
    texts_df
    .withColumn("chunks", chunk_text(F.col("text")))
    .select("doc_path", "doc_name", "chunks")
    .select("doc_path", "doc_name", F.posexplode("chunks").alias("chunk_num", "content"))
    .withColumn("chunk_id", F.md5(F.concat_ws("_", "doc_path", "chunk_num")))
    .withColumn("num_chars", F.length("content"))
    .withColumn("ingested_at", F.current_timestamp())
)

num_chunks = chunks_df.count()
num_docs = chunks_df.select("doc_path").distinct().count()
print(f"Produced {num_chunks} chunk(s) from {num_docs} document(s) with text content")

In [0]:
# Target table: primary key + change data feed so it is ready to back a
# Vector Search Delta Sync index later.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
    chunk_id     STRING NOT NULL,
    doc_path     STRING NOT NULL,
    doc_name     STRING,
    chunk_num    INT,
    content      STRING,
    num_chars    INT,
    ingested_at  TIMESTAMP,
    CONSTRAINT doc_chunks_pk PRIMARY KEY (chunk_id)
) TBLPROPERTIES (delta.enableChangeDataFeed = true);
""")

# Idempotent upsert: re-running the notebook refreshes chunks instead of
# duplicating them. WHEN NOT MATCHED BY SOURCE deletes target rows that no
# longer exist in the source — trailing chunks of documents that shrank and
# all chunks of documents removed from the volume.
chunks_df.createOrReplaceTempView("new_chunks")

spark.sql(f"""
MERGE INTO {CHUNK_TABLE} AS t
USING new_chunks AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

# --- Verification --------------------------------------------------------
print(f"Total rows in {CHUNK_TABLE}: {spark.table(CHUNK_TABLE).count()}")
display(
    spark.table(CHUNK_TABLE)
    .groupBy("doc_name")
    .agg(
        F.count("*").alias("chunks"),
        F.min("chunk_num").alias("first_chunk"),
        F.max("chunk_num").alias("last_chunk"),
        F.sum("num_chars").alias("total_chars"),
    )
)
display(spark.table(CHUNK_TABLE).orderBy("doc_name", "chunk_num").limit(5))